# 01 — Konsep Dasar Image Segmentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/04-segmentation/01_konsep_segmentasi.ipynb)

Object detection (modul 03) menghasilkan **kotak**. **Segmentation** lebih detail: menentukan kelas untuk **setiap piksel**, sehingga bentuk objek tergambar persis.

Isi notebook:
1. Tiga jenis segmentasi: **semantic**, **instance**, dan **panoptic**.
2. Representasi mask: peta indeks kelas, one-hot, mask biner per instance, **poligon** (format YOLO-seg), dan **RLE** (format COCO).
3. Metrik: pixel accuracy (dan jebakannya), **IoU per kelas**, **mIoU**, dan **Dice**, semuanya diimplementasikan sendiri.

Notebook ini hanya memakai NumPy dan Matplotlib, tanpa GPU.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1. Membuat Adegan Contoh

Kita buat gambar sintetis 128×128 berisi **dua kucing** (lingkaran) dan **satu mobil** (persegi panjang). Karena kita sendiri yang menggambarnya, kita tahu mask yang benar untuk setiap piksel (*ground truth*).

Kelas: `0 = background`, `1 = kucing`, `2 = mobil`.

In [ ]:
H, W = 128, 128
KELAS = ["background", "kucing", "mobil"]
yy, xx = np.mgrid[0:H, 0:W]  # koordinat setiap piksel

# Setiap instance: (kelas, mask boolean)
instances = [
    (1, (xx - 35) ** 2 + (yy - 40) ** 2 < 20 ** 2),       # kucing #1
    (1, (xx - 70) ** 2 + (yy - 50) ** 2 < 18 ** 2),       # kucing #2 (menempel dengan kucing #1)
    (2, (xx >= 60) & (xx < 115) & (yy >= 85) & (yy < 115)),  # mobil
]

# Gambar RGB sederhana: warna per instance + sedikit noise
image = np.full((H, W, 3), 0.85)
warna_instance = [(0.9, 0.5, 0.1), (0.6, 0.35, 0.1), (0.2, 0.4, 0.8)]
for (k, m), col in zip(instances, warna_instance):
    image[m] = col
image = np.clip(image + rng.normal(0, 0.03, image.shape), 0, 1)

plt.imshow(image); plt.title("Gambar contoh"); plt.axis("off"); plt.show()

## 2. Tiga Jenis Segmentasi

| Jenis | Pertanyaan yang dijawab | Output |
|---|---|---|
| **Semantic** | Piksel ini kelas apa? | Satu peta `H×W` berisi indeks kelas. Dua kucing yang menempel menjadi **satu area** "kucing". |
| **Instance** | Piksel ini milik objek yang mana? | Satu mask biner per objek (`N×H×W`) + kelasnya. Dua kucing **terpisah**. Background biasanya diabaikan. |
| **Panoptic** | Gabungan keduanya | Setiap piksel punya kelas **dan** ID instance. Objek yang bisa dihitung (*things*: kucing, mobil) dibedakan per instance; area yang tidak bisa dihitung (*stuff*: langit, jalan) hanya per kelas. |

In [ ]:
# Semantic: satu peta indeks kelas
semantic = np.zeros((H, W), dtype=np.int64)
for k, m in instances:
    semantic[m] = k

# Instance: tumpukan mask biner + daftar kelas
inst_masks = np.stack([m for _, m in instances])   # (N, H, W)
inst_kelas = np.array([k for k, _ in instances])

# Panoptic: ID unik per instance (0 = background)
panoptic = np.zeros((H, W), dtype=np.int64)
for i, m in enumerate(inst_masks, start=1):
    panoptic[m] = i

cmap_kelas = ListedColormap(["#eeeeee", "#e08a1e", "#3b6fd1"])
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
ax[0].imshow(image); ax[0].set_title("gambar")
ax[1].imshow(semantic, cmap=cmap_kelas, vmin=0, vmax=2); ax[1].set_title("semantic (warna = kelas)")
ax[2].imshow(image)
for i, m in enumerate(inst_masks):
    ax[2].contour(m, levels=[0.5], colors=[plt.cm.tab10(i)], linewidths=2)
ax[2].set_title(f"instance ({len(inst_masks)} objek)")
ax[3].imshow(panoptic, cmap="tab10", vmin=0, vmax=9); ax[3].set_title("panoptic (warna = ID instance)")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

print("shape semantic:", semantic.shape, "| nilai unik:", np.unique(semantic))
print("shape instance:", inst_masks.shape, "| kelas per instance:", inst_kelas)

## 3. Representasi Mask

### 3.1 Peta indeks vs one-hot

- **Peta indeks** `(H, W)` berisi angka kelas. Ini format label yang dipakai `nn.CrossEntropyLoss` untuk segmentasi.
- **One-hot** `(C, H, W)`: satu channel biner per kelas. Output model segmentasi juga berbentuk `(C, H, W)` (logits per kelas per piksel), lalu `argmax` di dimensi kelas menghasilkan peta indeks.

In [ ]:
one_hot = np.eye(len(KELAS), dtype=np.uint8)[semantic].transpose(2, 0, 1)  # (C, H, W)
print("one-hot shape:", one_hot.shape)
print("jumlah piksel per kelas:", {k: int(one_hot[i].sum()) for i, k in enumerate(KELAS)})
print("argmax kembali ke peta indeks sama?", np.array_equal(one_hot.argmax(0), semantic))

### 3.2 Poligon (format label YOLO-seg)

Menyimpan mask per piksel boros tempat. Format YOLO untuk segmentasi menyimpan **kontur objek sebagai poligon**, satu baris per objek:

```
<class_id> x1 y1 x2 y2 x3 y3 ...   (koordinat dinormalisasi 0–1)
```

Untuk lingkaran, kita bisa membuat poligon dengan mengambil titik-titik di kelilingnya.

In [ ]:
def lingkaran_ke_poligon(cx, cy, r, n_titik=16):
    sudut = np.linspace(0, 2 * np.pi, n_titik, endpoint=False)
    return np.stack([cx + r * np.cos(sudut), cy + r * np.sin(sudut)], axis=1)  # (n, 2) piksel


def poligon_ke_baris_yolo(kelas_id, poligon, w, h):
    # Normalisasi koordinat lalu ratakan menjadi x1 y1 x2 y2 ...
    norm = poligon / np.array([w, h])
    return f"{kelas_id} " + " ".join(f"{v:.4f}" for v in norm.ravel())


poli = lingkaran_ke_poligon(35, 40, 20)
# class_id di file label YOLO dimulai dari 0 untuk kelas objek (background tidak ditulis),
# jadi "kucing" = 0 dan "mobil" = 1 di file label.
baris = poligon_ke_baris_yolo(0, poli, W, H)
print(baris[:90], "...")
print(f"Jumlah angka: {len(baris.split())} (1 kelas + 16 titik × 2)")

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(inst_masks[0], cmap="gray")
ax.plot(*np.vstack([poli, poli[:1]]).T, "r.-")
ax.set_title("mask asli vs poligon 16 titik"); ax.axis("off"); plt.show()

### 3.3 RLE (Run-Length Encoding, format COCO)

RLE menyimpan mask biner sebagai **panjang deretan** nilai 0 dan 1 secara bergantian. Mask besar dengan area yang padat menjadi sangat ringkas.

Konvensi COCO: piksel dibaca **per kolom** (*column-major*, urutan Fortran), dan deretan pertama selalu menghitung angka **0**.

In [ ]:
def rle_encode(mask):
    piksel = mask.ravel(order="F").astype(np.uint8)       # baca per kolom
    batas = np.flatnonzero(np.diff(piksel)) + 1           # posisi nilai berubah
    panjang = np.diff(np.concatenate([[0], batas, [piksel.size]]))
    if piksel[0] == 1:
        panjang = np.concatenate([[0], panjang])          # deret pertama harus 0
    return {"size": list(mask.shape), "counts": panjang.tolist()}


def rle_decode(rle):
    h, w = rle["size"]
    nilai = np.zeros(sum(rle["counts"]), dtype=np.uint8)
    pos, isi = 0, 0
    for n in rle["counts"]:
        nilai[pos:pos + n] = isi
        pos, isi = pos + n, 1 - isi
    return nilai.reshape((h, w), order="F").astype(bool)


rle = rle_encode(inst_masks[2])   # mask mobil
print("RLE mobil (20 angka pertama):", rle["counts"][:20])
print(f"Panjang RLE: {len(rle['counts'])} angka vs {inst_masks[2].size} piksel")
print("Decode kembali identik?", np.array_equal(rle_decode(rle), inst_masks[2]))

### 3.4 Dari mask ke bounding box

Bounding box bisa diturunkan dari mask: ambil nilai min/max koordinat piksel yang bernilai 1. Inilah sebabnya model instance segmentation (Mask R-CNN, YOLO-seg) juga bisa menghasilkan kotak.

In [ ]:
def mask_ke_bbox(mask):
    ys, xs = np.nonzero(mask)
    return np.array([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])  # xyxy


for i, (k, m) in enumerate(instances):
    print(f"instance {i} ({KELAS[k]}): bbox xyxy = {mask_ke_bbox(m)} | luas mask = {m.sum()} piksel")

## 4. Metrik Segmentasi

Kita buat **prediksi palsu** yang meniru kesalahan model sungguhan:
- Tepi objek sedikit bergeser (erosi/dilasi).
- Sebagian kucing #2 salah diklasifikasi sebagai mobil.
- Ada bercak palsu di background.

In [ ]:
pred = semantic.copy()
pred[(xx - 38) ** 2 + (yy - 42) ** 2 < 20 ** 2] = 1        # kucing #1 bergeser 3 piksel
pred[(yy >= 50) & (yy < 68) & (xx >= 60) & (xx < 90) & (semantic == 1)] = 2  # sebagian kucing #2 → mobil
pred[(xx >= 60) & (xx < 115) & (yy >= 112) & (yy < 115)] = 0  # tepi bawah mobil hilang
pred[(xx - 110) ** 2 + (yy - 20) ** 2 < 6 ** 2] = 2          # bercak mobil palsu

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, data, judul in [(ax[0], semantic, "ground truth"), (ax[1], pred, "prediksi")]:
    a.imshow(data, cmap=cmap_kelas, vmin=0, vmax=2); a.set_title(judul); a.axis("off")
ax[2].imshow(semantic != pred, cmap="Reds"); ax[2].set_title("piksel salah"); ax[2].axis("off")
plt.tight_layout(); plt.show()

### 4.1 Pixel accuracy dan jebakannya

**Pixel accuracy** = persentase piksel yang benar. Sederhana, tetapi menyesatkan jika background mendominasi: model yang **selalu** memprediksi background pun bisa terlihat bagus.

In [ ]:
def pixel_accuracy(pred, gt):
    return (pred == gt).mean()


semua_background = np.zeros_like(semantic)
print(f"Pixel accuracy prediksi kita      : {pixel_accuracy(pred, semantic):.3f}")
print(f"Pixel accuracy 'semua background' : {pixel_accuracy(semua_background, semantic):.3f}  ← padahal tidak mendeteksi apa pun!")
print(f"Porsi background di gambar        : {(semantic == 0).mean():.3f}")

### 4.2 IoU per kelas, mIoU, dan Dice

Dihitung **per kelas** dengan menganggap piksel kelas itu sebagai himpunan:

$$\text{IoU}_c = \frac{|P_c \cap G_c|}{|P_c \cup G_c|} = \frac{TP}{TP + FP + FN} \qquad \text{Dice}_c = \frac{2|P_c \cap G_c|}{|P_c| + |G_c|} = \frac{2TP}{2TP + FP + FN}$$

- **mIoU** (*mean IoU*) = rata-rata IoU seluruh kelas. Ini metrik standar segmentasi semantik.
- **Dice** (sama dengan F1-score) selalu ≥ IoU, dengan hubungan $\text{Dice} = \frac{2\,\text{IoU}}{1 + \text{IoU}}$. Dice populer di citra medis, dan versi *soft*-nya dipakai sebagai loss (lihat notebook 02).

Cara efisien: bangun **confusion matrix** `C×C` sekali, lalu semua metrik diturunkan darinya.

In [ ]:
def confusion_matrix(pred, gt, n_kelas):
    # Baris = label asli, kolom = prediksi. bincount atas pasangan (gt, pred) yang dikodekan jadi satu angka.
    idx = gt.ravel() * n_kelas + pred.ravel()
    return np.bincount(idx, minlength=n_kelas ** 2).reshape(n_kelas, n_kelas)


def metrik_dari_cm(cm):
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp   # diprediksi kelas c padahal bukan
    fn = cm.sum(axis=1) - tp   # sebenarnya kelas c tapi terlewat
    iou = tp / np.maximum(tp + fp + fn, 1)
    dice = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    return iou, dice


cm = confusion_matrix(pred, semantic, len(KELAS))
iou, dice = metrik_dari_cm(cm)
print("Confusion matrix (baris = asli, kolom = prediksi):\n", cm, "\n")
print(f"{'kelas':<11} {'IoU':>6} {'Dice':>6} {'2IoU/(1+IoU)':>13}")
for k, i_, d in zip(KELAS, iou, dice):
    print(f"{k:<11} {i_:>6.3f} {d:>6.3f} {2 * i_ / (1 + i_):>13.3f}")
print(f"\nmIoU (3 kelas)        = {iou.mean():.3f}")
print(f"mIoU tanpa background = {iou[1:].mean():.3f}")

_, dice_bg = metrik_dari_cm(confusion_matrix(semua_background, semantic, len(KELAS)))
iou_bg, _ = metrik_dari_cm(confusion_matrix(semua_background, semantic, len(KELAS)))
print(f"\nmIoU 'semua background' = {iou_bg.mean():.3f}  ← jebakan pixel accuracy langsung terlihat")

### 4.3 IoU per instance

Untuk instance segmentation, IoU dihitung antara **mask prediksi dan mask ground truth per objek**, lalu dipakai untuk mencocokkan TP/FP seperti pada deteksi (modul 03). Hasilnya dirangkum dengan **mask mAP**. Rumusnya sama, hanya "kotak" diganti "mask".

In [ ]:
def mask_iou(a, b):
    irisan = np.logical_and(a, b).sum()
    gabungan = np.logical_or(a, b).sum()
    return irisan / gabungan if gabungan else 0.0


pred_kucing1 = (xx - 38) ** 2 + (yy - 42) ** 2 < 20 ** 2
print(f"Mask IoU kucing #1 (bergeser 3 piksel): {mask_iou(pred_kucing1, inst_masks[0]):.3f}")

# Bandingkan dengan IoU kotak untuk kasus yang sama
def box_iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    irisan = max(0, x2 - x1) * max(0, y2 - y1)
    luas = lambda k: (k[2] - k[0]) * (k[3] - k[1])
    return irisan / (luas(a) + luas(b) - irisan)

print(f"Box IoU kucing #1                     : {box_iou(mask_ke_bbox(pred_kucing1), mask_ke_bbox(inst_masks[0])):.3f}")

## Ringkasan

- **Semantic** = kelas per piksel; **instance** = mask per objek; **panoptic** = keduanya.
- Label semantik disimpan sebagai peta indeks `(H, W)`; output model berbentuk `(C, H, W)` lalu di-`argmax`.
- Instance mask bisa disimpan sebagai **poligon** (YOLO-seg) atau **RLE** (COCO).
- **Pixel accuracy** menyesatkan jika background dominan; pakai **IoU per kelas / mIoU** dan **Dice**.
- Semua metrik bisa diturunkan dari satu **confusion matrix** piksel.

## Latihan

1. **Boundary IoU.** IoU biasa kurang peka terhadap kesalahan di **tepi** objek besar. Implementasikan versi sederhana: ambil hanya piksel dalam jarak ≤ 2 piksel dari tepi mask (petunjuk: `mask & ~erosi(mask)` dengan erosi memakai `scipy.ndimage.binary_erosion`), lalu hitung IoU hanya di area tersebut. Bandingkan IoU biasa dan boundary IoU untuk prediksi kucing #1 yang bergeser 1, 3, dan 6 piksel.
2. **Mask → poligon.** Tulis fungsi yang mengubah mask biner menjadi poligon YOLO-seg memakai `cv2.findContours` (OpenCV terpasang di Colab), lalu kembalikan ke mask dengan `cv2.fillPoly`. Hitung IoU antara mask asli dan hasil rekonstruksi untuk 8, 16, dan 64 titik (petunjuk: `cv2.approxPolyDP`). Berapa titik yang cukup untuk IoU > 0.95?
3. **Metrik panoptic.** Pelajari definisi **Panoptic Quality**: `PQ = SQ × RQ`, dengan SQ = rata-rata IoU pasangan yang cocok dan RQ = F1 pencocokan pada IoU > 0.5. Implementasikan PQ untuk adegan di notebook ini dengan prediksi instance buatanmu sendiri (misalnya kucing #2 tidak terdeteksi dan ada satu instance palsu).